# Transforme sua AWS Lambda em MCP com Role IAM de Entrada no Gateway
## Transforme funções AWS Lambda em ferramentas MCP seguras com o Bedrock AgentCore Gateway

## Visão Geral
O Bedrock AgentCore Gateway oferece aos clientes uma forma de transformar suas funções AWS Lambda existentes em servidores MCP totalmente gerenciados, sem a necessidade de gerenciar infraestrutura ou hospedagem. O Gateway fornece uma interface uniforme do Model Context Protocol (MCP) para todas essas ferramentas. O Gateway emprega um modelo de autenticação dupla para garantir o controle de acesso seguro tanto para inbound requests quanto para outbound connections aos recursos de destino. O framework consiste em dois componentes principais: Inbound Auth (Inbound Auth), que valida e autoriza usuários que tentam acessar os alvos do gateway, e Outbound Auth (Outbound Auth), que permite ao gateway conectar-se com segurança aos recursos de backend em nome dos usuários autenticados. O Gateway utiliza uma role IAM para autorizar as chamadas às funções AWS Lambda para outbound auth.

Neste exemplo, demonstraremos tanto a inbound auth quanto outbound auth usando roles IAM.

![Como funciona](images/lambda-gw-iam-inbound.png)

### Detalhes do Tutorial


| Informação               | Detalhes                                                      |
|:-------------------------|:--------------------------------------------------------------|
| Tipo do tutorial         | Interativo                                                    |
| Componentes AgentCore    | AgentCore Gateway                                             |
| Framework de Agentes     | Strands Agents                                                |
| Tipo de alvo do Gateway  | AWS Lambda                                                    |
| Inbound Auth          | AWS IAM                                                       |
| Outbound Auth            | AWS IAM                                                       |
| Modelo LLM              | Anthropic Claude Haiku 4.5, Amazon Nova Pro                   |
| Componentes do tutorial  | Criação e Invocação do AgentCore Gateway                      |
| Vertical do tutorial     | Cross-vertical                                                |
| Complexidade do exemplo  | Fácil                                                         |
| SDK utilizado            | boto3                                                         |

Na primeira parte do tutorial, criaremos alvos (targets) do AgentCore Gateway para Lambda

### Arquitetura do Tutorial
Neste tutorial, transformaremos operações definidas em funções AWS Lambda em ferramentas MCP e as hospedaremos no Bedrock AgentCore Gateway. Demonstraremos a inbound auth usando credenciais AWS IAM no cabeçalho AWS Sigv4.
Para fins de demonstração, utilizaremos um Agente Strands usando modelos Amazon Bedrock.
No nosso exemplo, usaremos um agente muito simples com duas ferramentas: get_order e update_order.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Acesso ao Nova Pro via console AWS
* SDK do Amazon Bedrock AgentCore
* Strands Agents

## Configurando Autenticação para Requisições de Entrada do AgentCore Gateway
O AgentCore Gateway fornece conexões seguras via inbound auth e saída. Para a inbound auth, o AgentCore Gateway agora suporta credenciais/identidades AWS IAM, além do OAuth, para chamar o gateway. Se uma ferramenta precisa de acesso a recursos externos, o AgentCore Gateway pode usar outbound auth via API Key, IAM ou Token OAuth para permitir ou negar o acesso ao recurso externo.

Durante o fluxo de inbound auth, um agente ou o cliente MCP usa requisições assinadas com AWS Signature V4 que serão utilizadas para autenticação e autorização contra permissões IAM para acessar o AgentCore Gateway. O AgentCore Gateway então valida as credenciais/identidades AWS IAM e realiza a inbound auth.

Se a ferramenta executando no AgentCore Gateway precisa acessar recursos externos, a role IAM recuperará as credenciais dos recursos downstream para o alvo do Gateway. O AgentCore Gateway passa as credenciais de autorização ao chamador para obter acesso à API downstream.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os

#os.environ['AWS_ACCESS_KEY_ID']=''
#os.environ['AWS_SECRET_ACCESS_KEY']=''
os.environ['AWS_DEFAULT_REGION']='us-west-2' # set the AWS region




In [ ]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)



In [ ]:
# Now you can import utils
import utils
#### Create a sample AWS Lambda function that you want to convert into MCP tools
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")
if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda function created with ARN: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda function creation failed with message: ", lambda_resp['lambda_function_arn'])

In [ ]:
#### Create an IAM role for the Gateway to assume
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# Criar o Gateway com Autorizador Amazon IAM para inbound auth

In [ ]:
import time
import boto3
# CreateGateway with Amazon IAM. 
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])

create_response = gateway_client.create_gateway(name='TestGWforLambdaIAM',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='AWS_IAM',
    description='AgentCore Gateway with AWS Lambda target type using Amazon IAM for ingress auth'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)
time.sleep(10)

# Criar um alvo AWS Lambda e transformar em ferramentas MCP

## Recuperar a ARN da função Lambda
A célula abaixo busca a ARN da função Lambda criada anteriormente e armazena na variável `lambda_function_arn` para uso na configuração do target.

In [ ]:
import boto3

LAMBDA_FUNCTION_NAME = 'gateway_lambda'  # Nome da função Lambda criada anteriormente

lambda_client = boto3.client('lambda', region_name=os.environ['AWS_DEFAULT_REGION'])

try:
    response = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
    lambda_function_arn = response['Configuration']['FunctionArn']
    print(f"Lambda ARN encontrada: {lambda_function_arn}")
except Exception as e:
    print(f"Erro ao buscar a Lambda: {e}")
    lambda_function_arn = None

In [ ]:
# Configuração do target Lambda usando a ARN recuperada na célula anterior
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_function_arn, # ARN da Lambda recuperada automaticamente
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Lambda Target using SDK',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Criar Role AWS IAM para Invocar o Gateway 

A função abaixo cria ou atualiza uma role IAM que permite ao AWS Bedrock AgentCore invocar um gateway específico. Ela constrói e anexa uma política inline concedendo a permissão bedrock-agentcore:InvokeGateway para o ID do gateway fornecido. Além disso, configura uma política de confiança permitindo que tanto o serviço Bedrock AgentCore quanto a entidade IAM chamadora (current_arn) assumam a role.

In [ ]:
#### Create an IAM role to Invoke Gateway
current_role_arn = utils.get_current_role_arn()
print("Current role ARN: ", current_role_arn)

agentcore_gateway_iam_invoke_role = utils.create_gateway_invoke_tool_role("gateway-invoke-role",gatewayID , current_role_arn)
print("Role to invoke Agentcore gateway ARN: ", agentcore_gateway_iam_invoke_role['Role']['Arn'])

# Agente Strands chamando ferramentas MCP do AWS Lambda usando o Bedrock AgentCore Gateway

#### Suporte para Autenticação AWS IAM em SDKs de Cliente MCP

A autenticação AWS IAM agora é suportada para inbound requests no AgentCore Gateway. É importante notar que os SDKs de Cliente MCP de código aberto atuais têm suporte limitado para autenticação SigV4, particularmente para conexões HTTP com streaming. No entanto, a AWS forneceu uma solução através do projeto "Run Model Context Protocol (MCP) servers with AWS Lambda", que inclui uma extensão crucial para autenticação SigV4 com conexões HTTP de streaming.

Esta implementação, disponível no [repositório GitHub da AWS Labs](https://github.com/awslabs/run-model-context-protocol-servers-with-aws-lambda/tree/main), preenche a lacuna de autenticação para conexões de streaming e pode ser integrada perfeitamente com frameworks de agentes populares como Strands ou LangChain. A classe StreamableHTTPTransportWithSigV4 estende a camada de transporte MCP padrão para lidar com a assinatura AWS SigV4 enquanto mantém as capacidades de streaming, tornando-a compatível com o novo recurso de autenticação IAM do AgentCore Gateway.

In [ ]:
!pip3 install --upgrade strands-agents strands-agents-tools
from strands.models import BedrockModel

## The IAM credentials configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging
from strands import Agent
import logging
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client 
from botocore.credentials import Credentials
from streamable_http_sigv4 import (
    streamablehttp_client_with_sigv4,
)

SERVICE="bedrock-agentcore"

# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

def create_streamable_http_transport(mcp_url: str, access_token: str):
       return streamablehttp_client(mcp_url, headers={"Authorization": f"Bearer {access_token}"})

def create_streamable_http_transport_sigv4(mcp_url: str, key: str, secret: str, sessionToken: str, serviceName: str, awsRegion: str):
        iamcredentials = Credentials(
            access_key=key,
            secret_key=secret,
            token = sessionToken
        )
        return streamablehttp_client_with_sigv4(
            url=mcp_url,
            credentials=iamcredentials,
            service=serviceName,
            region=awsRegion,
        )

def get_full_tools_list(client):
    more_tools = True
    tools = []
    pagination_token = None
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)
        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True 
            pagination_token = tmp_tools.pagination_token
    return tools

def call_tool_sync(client, tool_id,tool_name, parameters=None):
    # Call the tool (no pagination argument supported)
    response = client.call_tool_sync(
        tool_use_id=tool_id,
        name=tool_name,
        arguments=parameters
    )

    # Extract output content
    if hasattr(response, "results") and response.results:
        return response.results
    elif hasattr(response, "output") and response.output:
        return response.output
    elif hasattr(response, "content"):
        return response.content
    else:
        return response  # fallback
         
def run_agent(mcp_url: str,key: str, secret: str, sessionToken: str,serviceName: str, awsRegion: str):
    mcp_client = MCPClient(lambda: create_streamable_http_transport_sigv4(mcp_url,key,secret,sessionToken,serviceName, awsRegion))

    with mcp_client:
        tools = get_full_tools_list(mcp_client)
        print(f"Found the following tools: {[tool.tool_name for tool in tools]}")
        print(f"Tool name: {tools[0].tool_name}")
        
        
        agent = Agent(model=yourmodel,tools=tools) ## you can replace with any model you like
        print(f"Tools loaded in the agent are {agent.tool_names}")
        agent("Check the order status for order id 123 and show me the exact response from the tool")
        #call mcp with tool
        tool=tools[0].tool_name
        tool_id="get-order-id-123-call-1"
        result = call_tool_sync(
            mcp_client,
            tool_id,
            tool_name=tool,
            parameters={"orderId": "123"}
        )

        print(f"Tool Call result: {result['content'][0]['text']}")
        

Assumir a role IAM de Invocação do Gateway e executar o Agente

In [ ]:
sts_client = boto3.client("sts")
response = sts_client.assume_role(
        RoleArn=agentcore_gateway_iam_invoke_role['Role']['Arn'],
        RoleSessionName="invoke_mcp_session",
        DurationSeconds=3600  # 1 hour, can be up to 12h for some roles
)

creds = response["Credentials"]

access=creds["AccessKeyId"]
secret=creds["SecretAccessKey"]
token=creds["SessionToken"]

# Run Agent with the credentials from this new gateway-invoke-role
time.sleep(10) 
run_agent(gatewayURL,access,secret,token, SERVICE, os.environ['AWS_DEFAULT_REGION'])

**Problema: se você receber o erro abaixo ao executar a célula abaixo, isso indica incompatibilidade entre as versões do pydantic e pydantic-core.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**Como resolver?**

Você precisará garantir que tenha pydantic==2.7.2 e pydantic-core 2.27.2, que são compatíveis entre si. Reinicie o kernel após a instalação.

# Limpeza

A célula abaixo deleta **todos** os recursos criados neste tutorial:
1. AgentCore Gateway (e seus targets)
2. Função AWS Lambda (`gateway_lambda`)
3. IAM Role da Lambda (`gateway_lambda_iamrole`)
4. IAM Role do Gateway (`agentcore-sample-lambdagateway-role`)
5. IAM Role de Invocação do Gateway (`gateway-invoke-role`)

## Excluir todos os recursos criados (Opcional)

In [ ]:
import boto3
import time
import utils
from botocore.exceptions import ClientError

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

def delete_iam_role_full(iam_client, role_name):
    """Deleta uma IAM Role removendo todas as policies antes."""
    try:
        # Desanexar managed policies
        attached = iam_client.list_attached_role_policies(RoleName=role_name)
        for policy in attached['AttachedPolicies']:
            iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy['PolicyArn'])
            print(f"   Desanexada policy: {policy['PolicyArn']}")
        # Deletar inline policies
        inline = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in inline['PolicyNames']:
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
            print(f"   Deletada inline policy: {policy_name}")
        # Deletar role
        iam_client.delete_role(RoleName=role_name)
        print(f"✅ IAM Role '{role_name}' deletada com sucesso.")
    except ClientError as e:
        if e.response['Error']['Code'] == 'NoSuchEntity':
            print(f"⚠️ IAM Role '{role_name}' já não existe.")
        else:
            print(f"⚠️ Erro ao deletar IAM Role '{role_name}': {e}")

# --- 1. Deletar o AgentCore Gateway (e seus targets) ---
print("=" * 60)
print("1. Deletando AgentCore Gateway e targets...")
print("=" * 60)
try:
    utils.delete_gateway(gateway_client, gatewayID)
    print(f"✅ Gateway {gatewayID} deletado com sucesso.")
except Exception as e:
    print(f"⚠️ Erro ao deletar gateway: {e}")

# --- 2. Deletar a função Lambda ---
print("\n" + "=" * 60)
print("2. Deletando função Lambda...")
print("=" * 60)
lambda_cleanup_client = boto3.client('lambda', region_name=REGION)
try:
    lambda_cleanup_client.delete_function(FunctionName='gateway_lambda')
    print("✅ Lambda 'gateway_lambda' deletada com sucesso.")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print("⚠️ Lambda 'gateway_lambda' já não existe.")
    else:
        print(f"⚠️ Erro ao deletar Lambda: {e}")

# --- 3. Deletar IAM Roles ---
print("\n" + "=" * 60)
print("3. Deletando IAM Roles...")
print("=" * 60)
iam_client = boto3.client('iam')

roles_to_delete = [
    'gateway_lambda_iamrole',
    'agentcore-sample-lambdagateway-role',
    'gateway-invoke-role',
]

for role_name in roles_to_delete:
    print(f"\n   Deletando role: {role_name}")
    delete_iam_role_full(iam_client, role_name)

print("\n" + "=" * 60)
print("🧹 Limpeza concluída!")
print("=" * 60)